In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

np.random.seed(42)

In [2]:
BASE_DIR = Path("..")

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
DOCS_DIR = BASE_DIR / "data" / "docs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
n_vehicles = 3000

models = ["Transit", "F-150", "E-Transit", "Super Duty", "Maverick"]
fuel_types = ["Gasoline", "Diesel", "Electric", "Hybrid"]
industries = ["Logistics", "Construction", "Utilities", "Delivery", "Government"]

vehicles = pd.DataFrame({
    "vehicle_id": [f"V{str(i).zfill(5)}" for i in range(1, n_vehicles + 1)],
    "customer_id": np.random.choice([f"C{str(i).zfill(4)}" for i in range(1, 501)], n_vehicles),
    "model": np.random.choice(models, n_vehicles),
    "year": np.random.randint(2016, 2026, n_vehicles),
    "fuel_type": np.random.choice(fuel_types, n_vehicles),
    "industry": np.random.choice(industries, n_vehicles),
    "mileage": np.random.randint(5_000, 180_000, n_vehicles)
})

vehicles["vehicle_age"] = 2026 - vehicles["year"]

vehicles.head()

,vehicle_id,customer_id,model,year,fuel_type,industry,mileage,vehicle_age
0,V00001,C0103,Transit,2018,Hybrid,Construction,125198,8
1,V00002,C0436,F-150,2020,Gasoline,Government,79155,6
2,V00003,C0349,Maverick,2020,Electric,Utilities,126449,6
3,V00004,C0271,F-150,2021,Diesel,Delivery,81700,5
4,V00005,C0107,Super Duty,2022,Hybrid,Utilities,11955,4


In [4]:
start_date = datetime(2026, 1, 1)
days = 120

records = []

for _, row in vehicles.iterrows():
    vehicle_id = row["vehicle_id"]
    age = row["vehicle_age"]
    mileage = row["mileage"]

    base_engine_temp = 85 + age * 1.2 + np.random.normal(0, 3)
    base_oil_pressure = 45 - age * 0.8 + np.random.normal(0, 2)
    base_battery_voltage = 12.6 - age * 0.05 + np.random.normal(0, 0.2)

    for d in range(days):
        date = start_date + timedelta(days=d)

        records.append({
            "vehicle_id": vehicle_id,
            "date": date,
            "engine_temp": base_engine_temp + np.random.normal(0, 5),
            "oil_pressure": base_oil_pressure + np.random.normal(0, 3),
            "battery_voltage": base_battery_voltage + np.random.normal(0, 0.3),
            "brake_events": np.random.poisson(8 + age * 0.4),
            "idle_hours": max(0, np.random.normal(2 + age * 0.1, 1)),
            "daily_miles": max(0, np.random.normal(75, 25))
        })

telematics = pd.DataFrame(records)
telematics.head()

,vehicle_id,date,engine_temp,oil_pressure,battery_voltage,brake_events,idle_hours,daily_miles
0,V00001,2026-01-01,93.350917,45.000877,12.134309,14,2.109864,69.096891
1,V00001,2026-01-02,97.449228,38.693877,12.657129,14,3.960258,31.999626
2,V00001,2026-01-03,99.705683,40.971823,12.256879,11,3.203803,66.622525
3,V00001,2026-01-04,96.828207,43.343437,12.079806,14,0.845895,71.984592
4,V00001,2026-01-05,94.173519,36.219813,12.477579,12,3.079657,104.067379


In [5]:
service_records = []

repair_types = ["Oil Change", "Brake Repair", "Battery Replacement", "Engine Inspection", "Transmission Repair", "Cooling System"]

for _, row in vehicles.iterrows():
    vehicle_id = row["vehicle_id"]
    age = row["vehicle_age"]

    n_services = np.random.poisson(1 + age * 0.4)

    for _ in range(n_services):
        service_date = start_date + timedelta(days=np.random.randint(0, days))

        repair_type = np.random.choice(
            repair_types,
            p=[0.35, 0.2, 0.15, 0.15, 0.08, 0.07]
        )

        base_cost = {
            "Oil Change": 120,
            "Brake Repair": 450,
            "Battery Replacement": 300,
            "Engine Inspection": 600,
            "Transmission Repair": 1800,
            "Cooling System": 700
        }[repair_type]

        service_records.append({
            "vehicle_id": vehicle_id,
            "service_date": service_date,
            "repair_type": repair_type,
            "repair_cost": round(base_cost * np.random.uniform(0.8, 1.4), 2)
        })

service_history = pd.DataFrame(service_records)
service_history.head()

,vehicle_id,service_date,repair_type,repair_cost
0,V00001,2026-02-20,Oil Change,101.57
1,V00001,2026-02-08,Battery Replacement,270.41
2,V00002,2026-01-01,Brake Repair,415.17
3,V00002,2026-03-13,Oil Change,131.46
4,V00002,2026-02-24,Oil Change,105.18


In [6]:
complaints = [
    "Vehicle overheating during route",
    "Battery warning light keeps appearing",
    "Brake noise reported by driver",
    "Transmission feels delayed",
    "Oil pressure warning triggered",
    "Vehicle idle time unusually high",
    "Reduced acceleration reported",
    "Check engine light is on",
    "No major issue reported"
]

ticket_records = []

for _, row in vehicles.iterrows():
    vehicle_id = row["vehicle_id"]
    age = row["vehicle_age"]

    n_tickets = np.random.poisson(0.5 + age * 0.2)

    for _ in range(n_tickets):
        complaint = np.random.choice(complaints)

        priority = "High" if any(x in complaint.lower() for x in ["overheating", "oil pressure", "transmission"]) else np.random.choice(["Low", "Medium"])

        ticket_records.append({
            "ticket_id": f"T{len(ticket_records)+1:06d}",
            "vehicle_id": vehicle_id,
            "ticket_date": start_date + timedelta(days=np.random.randint(0, days)),
            "complaint_text": complaint,
            "priority": priority
        })

support_tickets = pd.DataFrame(ticket_records)
support_tickets.head()

,ticket_id,vehicle_id,ticket_date,complaint_text,priority
0,T000001,V00001,2026-04-24,Oil pressure warning triggered,High
1,T000002,V00005,2026-01-17,No major issue reported,Medium
2,T000003,V00007,2026-01-05,Reduced acceleration reported,Low
3,T000004,V00007,2026-01-31,Oil pressure warning triggered,High
4,T000005,V00007,2026-04-22,No major issue reported,Medium


In [7]:
latest_telematics = telematics.groupby("vehicle_id").agg(
    avg_engine_temp=("engine_temp", "mean"),
    max_engine_temp=("engine_temp", "max"),
    avg_oil_pressure=("oil_pressure", "mean"),
    avg_battery_voltage=("battery_voltage", "mean"),
    brake_events_total=("brake_events", "sum"),
    idle_hours_total=("idle_hours", "sum")
).reset_index()

service_agg = service_history.groupby("vehicle_id").agg(
    repair_count=("repair_type", "count"),
    total_repair_cost=("repair_cost", "sum")
).reset_index()

ticket_agg = support_tickets.groupby("vehicle_id").agg(
    ticket_count=("ticket_id", "count"),
    high_priority_tickets=("priority", lambda x: (x == "High").sum())
).reset_index()

label_df = vehicles.merge(latest_telematics, on="vehicle_id", how="left")
label_df = label_df.merge(service_agg, on="vehicle_id", how="left")
label_df = label_df.merge(ticket_agg, on="vehicle_id", how="left")

label_df = label_df.fillna(0)

In [10]:
risk_score = (
    0.03 * label_df["vehicle_age"] +
    0.00001 * label_df["mileage"] +
    0.03 * (label_df["avg_engine_temp"] - 85) +
    -0.04 * (label_df["avg_oil_pressure"] - 40) +
    -0.20 * (label_df["avg_battery_voltage"] - 12) +
    0.10 * label_df["repair_count"] +
    0.20 * label_df["high_priority_tickets"] +
    np.random.normal(0, 0.5, len(label_df))
)

probability = 1 / (1 + np.exp(-risk_score))

failure_labels = pd.DataFrame({
    "vehicle_id": label_df["vehicle_id"],
    "failure_probability_true": probability,
    "failure_next_30_days": (probability > np.quantile(probability, 0.75)).astype(int)
})

failure_labels.head()

,vehicle_id,failure_probability_true,failure_next_30_days
0,V00001,0.927117,1
1,V00002,0.723535,0
2,V00003,0.911447,1
3,V00004,0.839943,0
4,V00005,0.704477,0


In [11]:
vehicles.to_csv(RAW_DIR / "vehicles.csv", index=False)
telematics.to_csv(RAW_DIR / "telematics.csv", index=False)
service_history.to_csv(RAW_DIR / "service_history.csv", index=False)
support_tickets.to_csv(RAW_DIR / "support_tickets.csv", index=False)
failure_labels.to_csv(RAW_DIR / "failure_labels.csv", index=False)

print("Saved raw datasets:")
print("vehicles:", vehicles.shape)
print("telematics:", telematics.shape)
print("service_history:", service_history.shape)
print("support_tickets:", support_tickets.shape)
print("failure_labels:", failure_labels.shape)

Saved raw datasets:
vehicles: (3000, 8)
telematics: (360000, 8)
service_history: (9500, 4)
support_tickets: (4721, 5)
failure_labels: (3000, 3)


In [9]:
docs = {
    "engine_manual.txt": """
Engine overheating may be caused by coolant leaks, radiator blockage, faulty thermostat,
water pump failure, low oil levels, or prolonged high-load operation. 
Recommended inspection steps include checking coolant level, radiator condition,
oil level, thermostat behavior, and engine temperature sensor readings.
""",

    "battery_manual.txt": """
Low battery voltage may indicate battery degradation, alternator issues,
loose connections, or excessive accessory load. Recommended inspection steps include
checking battery voltage, alternator output, terminal corrosion, and charging history.
""",

    "brake_manual.txt": """
Frequent brake events and brake noise may indicate worn brake pads, rotor issues,
hydraulic pressure problems, or aggressive driving behavior. Recommended inspection
includes checking brake pads, rotors, brake fluid, and driver event history.
""",

    "transmission_manual.txt": """
Transmission delay, slipping, or gear ratio errors may indicate low transmission fluid,
sensor faults, clutch wear, or control module issues. Recommended steps include checking
fluid level, diagnostic codes, transmission temperature, and prior service history.
""",

    "oil_pressure_manual.txt": """
Low oil pressure may be caused by low oil level, oil pump failure, clogged oil filter,
sensor malfunction, or engine wear. Recommended inspection includes checking oil level,
oil filter, pump pressure, and pressure sensor calibration.
"""
}

for filename, text in docs.items():
    with open(DOCS_DIR / filename, "w") as f:
        f.write(text)

print("Maintenance documents created.")

Maintenance documents created.
